## Modelo de Toomer 
1. NUcleos puntuales: represententan toda la masa de cada galaxia, es decir, las interacciones gravitacionales se dan con este y otros nucles.

2. Estrellas como particulas i,e la masa de las estrellas es m = 0 y esto quiere decir que no interactuan entre ellas.


In [2]:
import numpy as np
import matplotlib.pyplot as plt

from numba import jit

from typing import List, Dict, Any

In [9]:
class N_System:
    def __init__(self,particles: List[Dict[str, Any]]):
        '''Clase general para un sistema de N particulas,
        donde cada particula es un diccionario con sus propiedades.
        "mass": masa de la particula
        "position": posicion de la particula en el espacio (array de 3 dimensiones)
        "velocity": velocidad de la particula en el espacio (array de 3 dimensiones)
        '''
        self.n = len(particles)
        self.positions = np.array([p['position'] for p in particles])
        self.velocities = np.array([p['velocity'] for p in particles])
        self.masses = np.array([p['mass'] for p in particles])
        self.mass = self.masses.sum()

        v_prom = np.mean(np.linalg.norm(self.velocities, axis=1))
        self.t_relax = self.n / np.log(self.n) * (v_prom**3) / (self.mass*1.0) #tiempo de relajacion estimado

        ## indices de cuerpos con masa

        self.mass_indices = np.where(self.masses > 0)[0]

    def evolucionar(self, dt: float, T : float):
        '''integra el sistema por el método de Leapfrog durante un tiempo T con un paso dt
        
        parametros:
        dt: paso de tiempo para la integración
        T: tiempo total de evolución del sistema    

        returns:
        trayectorias: array de forma (n, num_steps, 3) con las posiciones de cada particula en cada paso de tiempo
        '''
    
        def acc(pos: np.ndarray,
                        masses: np.ndarray[float],
                        m_indices: np.ndarray[int],
                        n_particles: int) -> np.ndarray:
            '''Calcula la aceleración de cada particula debido a la gravedad con la fórmula de Newton'''
            acc = np.zeros_like(pos)
            for i in range(self.n):
                for j in m_indices:
                    if i  == j:
                        continue
                    r_vec = pos[i]- pos[j]
                    r_mag = np.linalg.norm(r_vec) + 1e-10 # para evitar division por cero
                    acc[i] -= masses[j] * r_vec / r_mag**3 
            return acc
    
        num_steps = int(T/dt)
        trajectories = np.zeros((self.n, num_steps, 3))

        #leapfrog integration
        accelerations = acc(self.positions, self.masses, self.mass_indices, self.n)
        self.velocities += 0.5 * accelerations * dt # kick inicial

        for step in range(1, num_steps):
            self.positions += self.velocities * dt # drift
            accelerations = acc(self.positions, self.masses, self.mass_indices, self.n) # calculo de aceleraciones
            self.velocities += accelerations * dt # kick
            trajectories[:, step, :] = self.positions
        return trajectories
            

In [4]:
mi_primer_sistema = N_System([
    {"mass": 1.0, "position": np.array([0.0, 0.0, 0.0]), "velocity": np.array([0.0, 0.0, 0.0])},
    {"mass": 2.0, "position": np.array([1.0, 0.0, 0.0]), "velocity": np.array([0.0, 1.0, 0.0])},
    {"mass": 3.0, "position": np.array([0.0, 1.0, 0.0]), "velocity": np.array([1.0, 0.0, 0.0])}
])  